# CyberShield Deecon Security Analyst Fine-Tuning + TFLite Policy Model

Bu notebook iki hedef icin hazirlandi:

1. `Zennar/Deecon-SecurityAnalyst-1.5B` modelini CyberShield olay formatina gore LoRA/QLoRA ile egitmek. Bu katman profesyonel aciklama, risk gerekcesi, false-positive notu ve guvenli mudahale onerisi uretir.
2. Android icinde calisacak kucuk `TFLite` policy modelini egitmek. Bu model metin uretmez; olaydan `allow/warn/block/quarantine/uninstall_prompt` gibi hizli karar uretir.

> Not: 1.5B LLM'i dogrudan TFLite olarak telefona koymak pratik degildir. Android tarafinda kullanilacak asil dosya kucuk policy `.tflite` modelidir. Deecon modeli ogretmen/analist katmani olarak kullanilir.

In [ ]:
# Colab runtime: GPU onerilir
# TensorFlow, NumPy ve Pandas Colab ortaminda zaten geliyor.
# Onlari zorla dusurmek paket cakismasi uretir; bu yuzden sadece LLM egitim paketlerini kuruyoruz.
# Pip kurulumundan hemen sonra TensorFlow import etmiyoruz; TFLite bolumunde kullanilacak.
!nvidia-smi || true
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub

import sys
print("Python:", sys.version)
print("Install cell finished. If Colab asks for runtime restart, restart and continue from the next cell.")


## Ayarlar

HF token gerektiren modeller icin Colab Secrets veya `huggingface-cli login` kullan. Public model ise indirme dogrudan calisabilir.

In [ ]:
import os, json, math, random, pathlib, textwrap
import numpy as np
import pandas as pd

BASE_MODEL = "Zennar/Deecon-SecurityAnalyst-1.5B"
OUTPUT_DIR = "/content/cybershield_deecon_lora"
TFLITE_DIR = "/content/cybershield_tflite_policy"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TFLITE_DIR, exist_ok=True)
SEED = 53
random.seed(SEED)
np.random.seed(SEED)

## CyberShield Model Katalogu

Repo icindeki `model_catalog.json` mantigi burada gomulu. Istersen GitHub'dan guncel katalog dosyasini indirerek degistirebilirsin.

In [ ]:
CATALOG = {
  "version": 1,
  "policy": {
    "mode": "automatic_defense",
    "battery_profile": "balanced",
    "max_hot_models": 2,
    "notification_opens_intervention_screen": true,
    "requires_user_confirmation_for_destructive_actions": true
  },
  "models": [
    {
      "id": "android_malware",
      "title": "Android Malware",
      "asset": "models/android_malware_detector.tflite",
      "input_size": 9503,
      "outputs": [
        "family_15",
        "malicious_probability"
      ],
      "threshold": 0.5,
      "accuracy": 0.9485200846,
      "recall": 0.9707430341,
      "interventions": [
        "warn",
        "quarantine_network",
        "uninstall_with_user_confirm"
      ]
    },
    {
      "id": "mirai",
      "title": "Mirai Malware",
      "asset": "models/mirai_manifest_detector.tflite",
      "input_size": 64,
      "outputs": [
        "category_8",
        "malicious_probability"
      ],
      "threshold": 0.705,
      "accuracy": 0.9962962963,
      "recall": 1.0,
      "interventions": [
        "warn",
        "block_network",
        "quarantine_network"
      ]
    },
    {
      "id": "network_attack",
      "title": "Network Attack",
      "asset": "models/network_attack_detector.tflite",
      "input_size": 79,
      "outputs": [
        "class_6"
      ],
      "threshold": 0.7231243849,
      "accuracy": 0.9584635083,
      "recall": 0.9800776386,
      "interventions": [
        "warn",
        "block_flow",
        "block_endpoint"
      ]
    },
    {
      "id": "dns_stateful",
      "title": "DNS Attack",
      "asset": "models/dns_stateful_binary_model.tflite",
      "input_size": 27,
      "outputs": [
        "attack_probability"
      ],
      "threshold": 0.5,
      "accuracy": 0.8126223092,
      "recall": 0.9936055091,
      "interventions": [
        "warn",
        "block_domain",
        "block_endpoint"
      ]
    },
    {
      "id": "doh_l1",
      "title": "DoH Detector",
      "asset": "models/l1_doh_nondoh_heuristic_model.tflite",
      "input_size": 29,
      "outputs": [
        "doh_probability"
      ],
      "threshold": 0.5248192549,
      "accuracy": 0.9923294808,
      "recall": 0.989148822,
      "interventions": [
        "route_to_doh_l2"
      ]
    },
    {
      "id": "doh_l2",
      "title": "Malicious DoH",
      "asset": "models/doh_heuristic_binary_model.tflite",
      "input_size": 29,
      "outputs": [
        "malicious_probability"
      ],
      "threshold": 0.2879624069,
      "accuracy": 0.9989685192,
      "recall": 0.9990650045,
      "interventions": [
        "warn",
        "block_flow",
        "block_endpoint"
      ]
    },
    {
      "id": "social_text",
      "title": "Social Engineering Text",
      "asset": "models/social_engineering_text_behavior_detector.tflite",
      "input_size": 2530,
      "outputs": [
        "malicious_probability"
      ],
      "threshold": 0.1315002143,
      "accuracy": 0.9778493758,
      "recall": 0.9621848739,
      "interventions": [
        "warn",
        "block_sender",
        "quarantine_message"
      ]
    },
    {
      "id": "social_url",
      "title": "Social Engineering URL",
      "asset": "models/social_engineering_url_feature_detector.tflite",
      "input_size": 48,
      "outputs": [
        "phishing_probability"
      ],
      "threshold": 0.261113584,
      "accuracy": 0.9766666667,
      "recall": 0.9786666667,
      "interventions": [
        "warn",
        "block_domain",
        "open_safe_browser_warning"
      ]
    },
    {
      "id": "phishing_html",
      "title": "Phishing HTML",
      "asset": "models/phishing_full_40.tflite",
      "input_size": 40,
      "outputs": [
        "phishing_probability"
      ],
      "threshold": 0.4,
      "accuracy": 0.90625,
      "recall": 0.922,
      "interventions": [
        "warn",
        "block_domain",
        "quarantine_message"
      ]
    },
    {
      "id": "iot_attack",
      "title": "IoT/IIoT Attack",
      "asset": "models/iot_iiot_attack_detector.tflite",
      "input_size": 71,
      "outputs": [
        "class_8"
      ],
      "threshold": 0.531388402,
      "accuracy": 0.9242097245,
      "recall": 0.9881932577,
      "interventions": [
        "warn",
        "block_endpoint",
        "isolate_device"
      ]
    },
    {
      "id": "attack_anomaly",
      "title": "TLS/Session Anomaly",
      "asset": "models/attack_anomaly_detector.tflite",
      "input_size": 32,
      "outputs": [
        "taxonomy_5",
        "anomaly_probability"
      ],
      "threshold": 0.5,
      "accuracy": 0.9733816546,
      "recall": 0.8273453094,
      "interventions": [
        "warn",
        "raise_risk_score"
      ]
    },
    {
      "id": "post_quantum",
      "title": "Post-Quantum Anomaly",
      "asset": "models/post_quantum_binary_anomaly_detector.tflite",
      "input_size": 32,
      "outputs": [
        "anomaly_probability"
      ],
      "threshold": 0.3958697617,
      "accuracy": 0.848787803,
      "recall": 0.9800399202,
      "interventions": [
        "warn",
        "block_flow",
        "explain_with_taxonomy"
      ]
    },
    {
      "id": "post_quantum_taxonomy",
      "title": "Post-Quantum Taxonomy",
      "asset": "models/post_quantum_taxonomy_classifier.tflite",
      "input_size": 32,
      "outputs": [
        "class_5"
      ],
      "threshold": 0.0,
      "accuracy": 0.8330417396,
      "recall": 0.0,
      "interventions": [
        "explain_only"
      ]
    },
    {
      "id": "post_quantum_subtype",
      "title": "Post-Quantum Subtype",
      "asset": "models/post_quantum_subtype_classifier.tflite",
      "input_size": 32,
      "outputs": [
        "class_12"
      ],
      "threshold": 0.0,
      "accuracy": 0.8197950512,
      "recall": 0.0,
      "interventions": [
        "explain_only"
      ]
    }
  ]
}

models = CATALOG["models"]
len(models), [m["id"] for m in models]

## Derin CyberShield olay verisi uretimi

Bu veri sadece yuzeysel skor/aksiyon degil; Deecon tarzina yakin olarak kanit, kok neden, etki, false-positive dusuncesi, guvenli aksiyon ve rollback bilgisi tasir.

In [ ]:
ACTION_TO_ID = {
    "allow": 0,
    "explain_only": 1,
    "warn": 2,
    "temporary_block": 3,
    "block_domain": 4,
    "block_ip": 5,
    "block_flow": 6,
    "quarantine": 7,
    "uninstall_prompt": 8,
}
ID_TO_ACTION = {v:k for k,v in ACTION_TO_ID.items()}
SOURCE_TO_ID = {
    "sms": 0, "shared_link": 1, "apk_monitor": 2, "vpn_dns": 3,
    "vpn_doh": 4, "vpn_flow": 5, "vpn_iot": 6, "vpn_tls": 7, "vpn_pqc": 8
}
TARGET_TO_ID = {"message":0, "url":1, "domain":2, "ip":3, "flow":4, "apk":5, "device":6, "session":7}

DOMAIN_PROFILES = {
    "android_malware": ("apk_monitor", "apk", "malware_installation", "uninstall_prompt"),
    "mirai": ("vpn_iot", "device", "botnet_mirai", "quarantine"),
    "network_attack": ("vpn_flow", "flow", "network_intrusion", "block_flow"),
    "dns_stateful": ("vpn_dns", "domain", "dns_attack", "block_domain"),
    "doh_l1": ("vpn_doh", "flow", "encrypted_dns_detection", "explain_only"),
    "doh_l2": ("vpn_doh", "flow", "malicious_doh", "block_flow"),
    "social_text": ("sms", "message", "social_engineering", "quarantine"),
    "social_url": ("shared_link", "url", "phishing_url", "block_domain"),
    "phishing_html": ("shared_link", "url", "phishing_page", "block_domain"),
    "iot_attack": ("vpn_iot", "device", "iot_intrusion", "quarantine"),
    "attack_anomaly": ("vpn_tls", "session", "tls_session_anomaly", "warn"),
    "post_quantum": ("vpn_pqc", "session", "post_quantum_anomaly", "block_flow"),
    "post_quantum_taxonomy": ("vpn_pqc", "session", "post_quantum_taxonomy", "explain_only"),
    "post_quantum_subtype": ("vpn_pqc", "session", "post_quantum_subtype", "explain_only"),
}

TARGETS = {
    "domain": ["secure-update-check.example", "cdn-login-verify.example", "dns-tunnel-node.example"],
    "url": ["https://secure-login-verify.example/account", "https://wallet-recovery.example/claim"],
    "ip": ["203.0.113.45", "198.51.100.77"],
    "flow": ["203.0.113.45:443", "198.51.100.77:53"],
    "apk": ["com.fake.cleaner", "com.security.update.service"],
    "device": ["iot-camera-lan", "router-lan-client"],
    "message": ["sms:bank-verify", "sms:urgent-prize"],
    "session": ["tls-session-443", "pqc-handshake-session"]
}

def choose_action(base_action, risk, model_id, false_positive_hint=False):
    if model_id in ("doh_l1", "post_quantum_taxonomy", "post_quantum_subtype"):
        return "explain_only"
    if risk < 0.25:
        return "allow" if false_positive_hint else "explain_only"
    if risk < 0.50:
        return "warn"
    if risk < 0.75:
        return "temporary_block" if base_action.startswith("block") else "warn"
    return base_action

def severity_for(risk):
    if risk >= 0.85: return "critical"
    if risk >= 0.65: return "high"
    if risk >= 0.40: return "medium"
    if risk >= 0.20: return "low"
    return "informational"

def evidence_for(threat_type, target_type):
    mapping = {
        "malicious_doh": ["encrypted DNS channel", "DoH-like HTTPS flow", "malicious probability above calibrated threshold"],
        "dns_attack": ["abnormal DNS query pattern", "domain entropy and repeated lookup evidence", "stateful DNS model exceeded threshold"],
        "malware_installation": ["suspicious permissions", "new APK installation event", "manifest and component risk indicators"],
        "social_engineering": ["urgency and fear language", "credential or OTP request", "external link in message"],
        "phishing_url": ["login/verify keywords", "suspicious host/path structure", "phishing probability above threshold"],
        "phishing_page": ["credential form indicators", "script or iframe risk", "mixed-content or impersonation signal"],
        "network_intrusion": ["flow rate anomaly", "destination endpoint risk", "packet and byte pattern deviation"],
        "iot_intrusion": ["IoT endpoint behavior anomaly", "botnet-like traffic pattern", "device isolation may reduce spread"],
        "botnet_mirai": ["Mirai-like category signal", "IoT malware recall is high", "network quarantine recommended"],
        "tls_session_anomaly": ["TLS/session anomaly score", "unusual encrypted session behavior"],
        "post_quantum_anomaly": ["PQC/TLS session anomaly", "taxonomy model can explain subtype"],
    }
    return mapping.get(threat_type, ["model confidence", f"target type {target_type}", "policy threshold evidence"])

def explanation(event):
    action = event["recommended_action"]
    risk_pct = int(round(event["risk_score"] * 100))
    ev = ", ".join(event["evidence"][:2])
    fp = event["false_positive_note"]
    return (
        f"CyberShield classified this {event['target_type']} event as {event['threat_type']} with {risk_pct}% risk. "
        f"The main evidence is {ev}. Recommended action is {action} because it reduces exposure while keeping the user in control. "
        f"False-positive consideration: {fp} Rollback: {event['rollback']}"
    )

def make_event(model, risk, variant):
    model_id = model["id"]
    source, target_type, threat_type, base_action = DOMAIN_PROFILES[model_id]
    false_positive_hint = variant % 5 == 0
    action = choose_action(base_action, risk, model_id, false_positive_hint)
    target = random.choice(TARGETS[target_type])
    severity = severity_for(risk)
    evidence = evidence_for(threat_type, target_type)
    fp_note = "Legitimate enterprise or privacy tools can produce similar signals." if false_positive_hint else "No strong benign context is known for this target."
    rollback = "Remove the target from the temporary block or allow list after review."
    event = {
        "source": source,
        "detector_model": model_id,
        "detector_title": model["title"],
        "risk_score": round(float(risk), 4),
        "threshold": model.get("threshold", 0.5),
        "model_accuracy": model.get("accuracy", 0.0),
        "model_recall": model.get("recall", 0.0),
        "target_type": target_type,
        "target": target,
        "threat_type": threat_type,
        "severity": severity,
        "evidence": evidence,
        "false_positive_note": fp_note,
        "recommended_action": action,
        "requires_user_confirmation": action in ["block_domain", "block_ip", "block_flow", "quarantine", "uninstall_prompt", "temporary_block"],
        "rollback": rollback,
    }
    event["analyst_response"] = explanation(event)
    return event

risks = [0.08, 0.18, 0.31, 0.46, 0.58, 0.71, 0.84, 0.93, 0.985]
events = []
for model in models:
    for i in range(36):
        base = risks[i % len(risks)]
        jitter = random.uniform(-0.035, 0.035)
        risk = min(0.999, max(0.001, base + jitter))
        events.append(make_event(model, risk, i))

random.shuffle(events)
len(events), events[0]


In [ ]:
# SFT JSONL ve policy CSV kaydet
from pathlib import Path
out = Path('/content/cybershield_training_data')
out.mkdir(exist_ok=True)

sft_path = out / 'cybershield_deecon_sft.jsonl'
policy_path = out / 'cybershield_policy_dataset.csv'

def to_prompt(event):
    inp = {k:v for k,v in event.items() if k != 'analyst_response'}
    return (
        "You are CyberShield's cybersecurity incident analyst. "
        "Analyze the structured mobile security event, explain the risk, consider false positives, "
        "and recommend the safest user-approved response.\n\n"
        "### CyberShield Event\n" + json.dumps(inp, indent=2) + "\n\n### Analyst Response\n"
    )

with open(sft_path, 'w', encoding='utf-8') as f:
    for ev in events:
        row = {
            "messages": [
                {"role":"system", "content":"You are a defensive cybersecurity analyst for the CyberShield Android app. Never provide offensive instructions. Focus on safe explanation, user-approved response, rollback, and false-positive awareness."},
                {"role":"user", "content":to_prompt(ev)},
                {"role":"assistant", "content":ev['analyst_response']}
            ]
        }
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

rows = []
for ev in events:
    rows.append({
        "source_id": SOURCE_TO_ID[ev['source']],
        "model_id": [m['id'] for m in models].index(ev['detector_model']),
        "target_type_id": TARGET_TO_ID[ev['target_type']],
        "risk_score": ev['risk_score'],
        "threshold": ev['threshold'],
        "over_threshold": float(ev['risk_score'] >= ev['threshold']),
        "accuracy": ev['model_accuracy'],
        "recall": ev['model_recall'],
        "requires_user_confirmation": float(ev['requires_user_confirmation']),
        "severity_id": ["informational","low","medium","high","critical"].index(ev['severity']),
        "action_id": ACTION_TO_ID[ev['recommended_action']],
    })
pd.DataFrame(rows).to_csv(policy_path, index=False)
print(sft_path)
print(policy_path)
print(pd.DataFrame(rows).head())

## Deecon 1.5B LoRA fine-tuning

Bu bolum GPU gerektirir. Colab T4 ile kucuk epoch/small batch baslangic icin uygundur. Daha guclu GPU varsa epoch ve veri miktari artirilabilir.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

dataset = load_dataset('json', data_files=str(sft_path), split='train')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_chat(example):
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(example['messages'], tokenize=False)
    return '\n'.join([m['role'].upper() + ': ' + m['content'] for m in example['messages']])

def add_text(example):
    example['text'] = format_chat(example)
    return example

dataset = dataset.map(add_text)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    fp16=True,
    optim='paged_adamw_8bit',
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=1024,
    args=args,
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved LoRA adapter:', OUTPUT_DIR)

## H?zl? Deecon test prompt'u

In [ ]:
test_event = events[0]
prompt = to_prompt(test_event)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=180, temperature=0.2, do_sample=True)
print(tokenizer.decode(output[0], skip_special_tokens=True)[-1200:])

## Android icin kucuk TFLite Policy Model

Bu model CyberShield uygulamasi icine konulacak asil hafif karar modelidir. Deecon tarzindaki olay verisinden `recommended_action` sinifini ogrenir.

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import joblib

df = pd.read_csv(policy_path)
X = df.drop(columns=['action_id']).astype('float32').values
y = df['action_id'].astype('int32').values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=SEED, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype('float32')
X_test_s = scaler.transform(X_test).astype('float32')

policy_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_s.shape[1],), name='cybershield_event_features'),
    tf.keras.layers.Dense(96, activation='relu'),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(ACTION_TO_ID), activation='softmax', name='intervention_action'),
])
policy_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = policy_model.fit(X_train_s, y_train, validation_data=(X_test_s, y_test), epochs=80, batch_size=24, verbose=1)

pred = policy_model.predict(X_test_s).argmax(axis=1)
print('accuracy', accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=[ID_TO_ACTION[i] for i in range(len(ID_TO_ACTION))]))

saved_model_dir = f'{TFLITE_DIR}/saved_model'
policy_model.export(saved_model_dir)
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = f'{TFLITE_DIR}/cybershield_policy_intervention_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

metadata = {
    'feature_order': ['source_id','model_id','target_type_id','risk_score','threshold','over_threshold','accuracy','recall','requires_user_confirmation','severity_id'],
    'actions': ID_TO_ACTION,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'base_teacher_model': BASE_MODEL,
    'purpose': 'CyberShield on-device intervention recommendation policy model',
    'note': 'This TFLite model recommends defensive user-approved actions; it does not generate text.'
}
meta_path = f'{TFLITE_DIR}/cybershield_policy_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(tflite_path)
print(meta_path)

## Dosyalari indir

Colab sol panelden `/content/cybershield_tflite_policy/` klasorunu indirebilirsin. LoRA adapter icin `/content/cybershield_deecon_lora/` klasorunu indir veya Hugging Face'e yukle.

In [ ]:
!ls -lh /content/cybershield_tflite_policy
!zip -r /content/cybershield_tflite_policy.zip /content/cybershield_tflite_policy
!zip -r /content/cybershield_deecon_lora.zip /content/cybershield_deecon_lora
print('/content/cybershield_tflite_policy.zip')
print('/content/cybershield_deecon_lora.zip')